# Kubernetes without a cluster

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 35 §35.2–§35.4 · type: concept-lab*

**The promise:** by the end you can read real Kubernetes manifests fluently, explain the declarative *desired-state* reconcile loop and HPA autoscaling by **simulating both offline**, and defend a **Fargate-vs-Kubernetes** decision on your team's real constraints — all without standing up a cluster.

## 🧠 Why this matters

Kubernetes' one big idea is **declarative, desired-state** management: you *declare* the state you want — "keep 3 replicas of this Pod" — and a controller continuously works to make reality match, restarting, rescheduling, and rebalancing without you in the loop. Everything else (Services, Ingress, HPA, probes) hangs off that idea.

You don't need a cluster to *understand* it — and understanding it is what lets you make the call the book cares about most: **when not to reach for K8s.** A cluster is expensive to operate (networking, storage, upgrades, security — a near-full-time job). So we learn the model the cheap way: read the YAML, simulate the reconcile loop and the autoscaler as a few lines of Python, and spend the saved effort on the *decision*. See §35.2–§35.4.

## Objectives & prereqs

**By the end you can:**
- Read a `Deployment` / `Service` / `Ingress` / `HPA` / `ConfigMap` / `Secret` and say what each object's *job* is.
- Explain **desired-state reconciliation**: declare N replicas → the controller makes reality match.
- Compute **HPA** target replicas from a load metric (queue depth) and find the scale-up point.
- Model **readiness vs. liveness** probes as a small state machine (gate traffic vs. restart).
- Defend a **Fargate-vs-Kubernetes** choice by modeling operational *cost*, not just capability.

**Prereqs:** 35-01 (the image these manifests deploy). Ch 28 (liveness/readiness probes). Ch 33 (Fargate/EKS, ECR trade-off). Ch 31 (Celery workers + queue depth, for the HPA example).

**Runs free & offline.** No cluster, no cloud, no API key, no spend. The manifests in `data/` are **parsed and simulated, never applied** — there is no `kubectl` anywhere in this notebook.

In [ ]:
# Setup — imports, env, the MOCK switch, and a seed. Pure stdlib; fully offline.
import os
import random
import re
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# MOCK=1 (default) is the only path this notebook needs: everything here is a local
# simulation. There is NO live/cluster path — manifests are never applied. The switch
# exists for consistency with the rest of the course and to assert "no cluster touched."
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Seed every stochastic step so the reconcile/scaling simulations are reproducible.
random.seed(35)

# Where the committed manifest fixtures live (small, readable, real K8s YAML).
DATA = Path("data")

print("MOCK mode :", MOCK, "— offline simulation; no cluster, no kubectl, no spend")
print("manifests :", ", ".join(sorted(p.name for p in DATA.glob("*.yaml"))))
print("seed      : 35  (reconcile + HPA sims are deterministic)")

## Read the manifests: name each object's job

Six small YAML files in `data/` describe one agent service the K8s way. We don't need a YAML library to *read* them — we'll display each file and extract just the two fields that identify an object (`kind` and `metadata.name`) with a couple of stdlib regexes. (Real tooling parses the full tree; here we want you reading the YAML with your eyes, not hiding it behind a parser.)

In [ ]:
# A deliberately tiny, dependency-free reader: pull `kind` and the FIRST `name:` (the
# object's metadata.name) from a manifest's text. Enough to label each object's job.
def identify(text):
    kind = re.search(r'^kind:\s*(\S+)', text, re.MULTILINE)
    name = re.search(r'^\s+name:\s*(\S+)', text, re.MULTILINE)  # first indented name == metadata.name
    return (kind.group(1) if kind else "?"), (name.group(1) if name else "?")


JOBS = {
    "Deployment": "keep N replicas of a Pod running; roll updates (the desired-state engine)",
    "Service":    "stable address + load-balance across the Deployment's Pods",
    "Ingress":    "admit external HTTP traffic and route it to a Service (the front door)",
    "HorizontalPodAutoscaler": "add/remove replicas to track a metric (here: queue depth)",
    "ConfigMap":  "non-secret config, injected as env vars",
    "Secret":     "sensitive values injected at runtime — never baked into the image",
}

for path in sorted(DATA.glob("*.yaml")):
    kind, name = identify(path.read_text(encoding="utf-8"))
    print(f"{path.name:18} {kind:24} name={name:16} -> {JOBS.get(kind, '?')}")

Read one in full — the `Deployment`, the heart of the model. Notice three things: `replicas: 3` (the *desired* count the controller defends), the `readinessProbe`/`livenessProbe` (Ch 28, simulated later), and `envFrom` pulling a `ConfigMap` and a `Secret` — config and secrets injected at runtime, never baked into the image (§35.1).

In [ ]:
print((DATA / "deployment.yaml").read_text(encoding="utf-8"))

### 🔮 Predict: the reconcile loop

A `Deployment` says `replicas: 3`. The controller's whole job is to make the number of *running, healthy* Pods equal that **desired** number — continuously. If reality drifts (a Pod crashes, a node dies), it acts: start a Pod when there are too few, delete one when there are too many.

We'll model that as a one-line rule: `delta = desired - running`; `delta > 0` → **start** Pods, `delta < 0` → **delete** Pods, `delta == 0` → **do nothing**.

**Predict:** desired = 3. A node fails and takes **one** Pod with it, so running drops to 2. What does the controller do on its next pass — and what's the running count after it acts?

In [ ]:
# A minimal desired-state reconciler. This is the entire idea of Kubernetes, distilled.
def reconcile(desired, running):
    delta = desired - running
    if delta > 0:
        return f"START  {delta} pod(s)", running + delta
    if delta < 0:
        return f"DELETE {-delta} pod(s)", running + delta   # delta is negative
    return "no-op (converged)", running


desired = 3
running = 3
print(f"desired = {desired}\n")

# A Pod dies (node failure). The controller notices and reconciles.
running = 2
print(f"event: a node fails -> running drops to {running}")
action, running = reconcile(desired, running)
print(f"  action: {action:16} -> running now {running}")

# Next pass: already converged, so it does nothing.
action, running = reconcile(desired, running)
print(f"  action: {action:16} -> running now {running}")

assert running == desired, "controller must drive running back to desired"
print("\nConverged: running == desired. No human paged.")

**What you just saw.** You never told Kubernetes "restart the pod." You declared `replicas: 3`; the controller observed `running == 2`, computed the gap, and **started one** — then, on the next pass, saw the gap was closed and did nothing. *Self-healing* is just this loop running forever. Scaling is the same rule with a different `desired`: change it to 5 and the very next pass starts two more.

### Simulate the HPA: scale workers to queue depth

The `HorizontalPodAutoscaler` (`data/hpa.yaml`) automates the `desired` count itself. Its §35.3 job here: scale Celery-style workers (Ch 31) to the **backlog** in the broker. The rule the HPA actually uses:

> `desiredReplicas = ceil(currentReplicas × currentMetric / targetMetric)`

Our `targetMetric` is `averageValue: 30` queued tasks per worker, clamped to `minReplicas: 1 .. maxReplicas: 10`. We'll feed a synthetic backlog curve (a morning spike) and watch the autoscaler track it.

In [ ]:
import math

# HPA parameters straight from data/hpa.yaml (no YAML lib needed — read them off the file).
TARGET_PER_POD = 30   # averageValue: tasks per worker we aim to hold
MIN_REPLICAS = 1
MAX_REPLICAS = 10


def hpa_desired(current_replicas, total_queue_depth):
    """The real HPA ratio rule, clamped to [min, max]."""
    current_per_pod = total_queue_depth / max(current_replicas, 1)
    raw = math.ceil(current_replicas * current_per_pod / TARGET_PER_POD)
    return max(MIN_REPLICAS, min(MAX_REPLICAS, raw))


# A synthetic morning backlog curve: queue depth (tasks waiting) sampled each minute.
# Deterministic shape + a little seeded jitter so it reads like real telemetry.
base = [10, 25, 60, 150, 240, 300, 270, 180, 90, 40, 20, 10]
queue_curve = [max(0, v + random.randint(-8, 8)) for v in base]
print("minute :", " ".join(f"{i:4d}" for i in range(len(queue_curve))))
print("backlog:", " ".join(f"{v:4d}" for v in queue_curve))

**🔮 Predict:** the curve peaks around **300** queued tasks, the target is **30** per worker, and `maxReplicas` is **10**. At the peak, how many worker Pods will the HPA ask for — exactly 10, or fewer? (Hint: 300 / 30 = 10, but the cap is also 10.) And at what minute does it first hit the cap?

In [ ]:
replicas = MIN_REPLICAS
first_cap_minute = None
print(f"{'min':>3}  {'backlog':>7}  {'replicas':>8}  {'per-pod':>7}")
for minute, depth in enumerate(queue_curve):
    replicas = hpa_desired(replicas, depth)
    per_pod = depth / replicas
    flag = ""
    if replicas == MAX_REPLICAS and first_cap_minute is None:
        first_cap_minute = minute
        flag = "  <-- hit maxReplicas (backlog now grows unbounded; raise the cap or shed load)"
    print(f"{minute:>3}  {depth:>7}  {replicas:>8}  {per_pod:>7.1f}{flag}")

print(f"\nFirst hit maxReplicas at minute {first_cap_minute}.")
print("Below the cap the HPA holds ~30 tasks/pod; at the cap, extra backlog just queues.")

**What you just saw.** The autoscaler tracks load by *ratio*, holding roughly `TARGET_PER_POD` tasks on each worker — until it slams into `maxReplicas`. Past that point adding load no longer adds workers; the backlog grows and latency climbs. That cap is a deliberate cost guardrail (Ch 31), and watching *where* you hit it is how you size `maxReplicas` and decide when to shed load instead.

### Health probes as a state machine (Ch 28)

Two probes, two **different** jobs — confusing them is a classic outage:

- **readiness** decides whether the Pod gets *traffic*. Fail it → the Service stops routing to this Pod (but it keeps running).
- **liveness** decides whether the Pod gets *restarted*. Fail it → the kubelet kills and recreates the Pod.

We'll model one Pod's lifecycle as a tiny state machine driven by probe results.

In [ ]:
def pod_state(ready, alive):
    """Map (readiness, liveness) probe results to what Kubernetes does."""
    if not alive:
        return "RESTART  (liveness failed -> kill & recreate the Pod)"
    if not ready:
        return "NO-TRAFFIC (readiness failed -> stays up, Service routes around it)"
    return "SERVING  (ready & alive -> receives traffic)"


# A Pod's first 25s: slow boot (not ready), then healthy, then a hang (liveness fails).
timeline = [
    (0,  False, True),   # booting: alive but not yet ready -> held out of rotation
    (3,  False, True),   # still warming up
    (6,  True,  True),   # ready! -> starts serving
    (12, True,  True),   # steady state
    (18, False, False),  # hung: both probes fail -> restart
    (21, False, True),   # rebooted, warming up again
    (24, True,  True),   # ready again
]
for t, ready, alive in timeline:
    print(f"t={t:>2}s  ready={str(ready):5}  alive={str(alive):5}  -> {pod_state(ready, alive)}")

**What you just saw.** Readiness *gates traffic without killing* — a Pod that's warming up (or briefly overloaded) is simply held out of rotation, so users never hit it. Liveness *restarts* a genuinely stuck Pod. Wiring both correctly (against the cheap `/healthz` from 35-01) is what makes the reconcile loop's self-healing actually work; get them backwards and you either restart healthy-but-busy Pods or send traffic to dead ones.

### Helm & operators (conceptual — nothing deployed)

Raw manifests get repetitive across environments (dev/staging/prod differ by a few values). Two ecosystem pieces (§35.4), described, not run:

- **Helm** is the package manager: templated, versioned bundles called **charts**. You write the manifest *once* with `{{ .Values.replicas }}` holes and supply a small `values.yaml` per environment — the same chart, parameterized.
- **Operators** extend Kubernetes with **custom controllers** that encode operational know-how for a specific system (a database, a model server, a vector store), automating *day-two* work — upgrades, backups, failover — the way an expert on-call would.

Both are leverage *and* more surface to operate — which is exactly the trade-off the next cells make you weigh.

In [ ]:
# A 30-second feel for Helm templating: one chart, two environments, zero new YAML.
def render_chart(template, values):
    """Toy Go-template-style substitution: {{ .Values.key }} -> values[key]."""
    out = template
    for key, val in values.items():
        out = out.replace("{{ .Values." + key + " }}", str(val))
    return out


chart = "replicas: {{ .Values.replicas }}\nimageTag: {{ .Values.imageTag }}"
for env, values in {
    "dev":  {"replicas": 1, "imageTag": "dev"},
    "prod": {"replicas": 5, "imageTag": "1.4.2"},
}.items():
    print(f"# values-{env}.yaml ->")
    print(render_chart(chart, values))
    print()

### ⚠️ Pitfall: adopting Kubernetes when Fargate would do (§35.2)

The most common infrastructure mistake in our field is reaching for Kubernetes for a workload that **Fargate or two containers** would serve perfectly (Ch 33). K8s' power is real — and so is its **permanent operational cost**: networking, storage, security, upgrades, a steep learning curve, often a dedicated platform person.

Let's make the decision *quantitative* instead of aspirational: score your real constraints and let the model recommend.

In [ ]:
# A small decision aid. Each factor pushes toward managed containers (Fargate) or K8s.
# Score your situation honestly; the tool tallies — it does not flatter the shiny option.
def recommend(*, services, teams, needs_gpu_scheduling, needs_multicloud,
              has_platform_engineer, traffic_is_spiky_but_simple):
    k8s_points = 0
    reasons = []
    if services >= 10:
        k8s_points += 2; reasons.append("many services to orchestrate (+2 K8s)")
    if teams >= 3:
        k8s_points += 1; reasons.append("multiple teams sharing infra (+1 K8s)")
    if needs_gpu_scheduling:
        k8s_points += 2; reasons.append("GPU scheduling across a pool (+2 K8s)")
    if needs_multicloud:
        k8s_points += 2; reasons.append("multi-cloud portability (+2 K8s)")
    if has_platform_engineer:
        k8s_points += 1; reasons.append("someone can actually operate it (+1 K8s)")
    if traffic_is_spiky_but_simple and services < 10:
        k8s_points -= 1; reasons.append("simple service, just spiky -> Fargate scales it (-1 K8s)")

    verdict = "Kubernetes (you genuinely need it)" if k8s_points >= 3 else \
              "Managed containers / Fargate (ship faster, operate less)"
    return verdict, k8s_points, reasons


# Scenario: a 2-person team shipping ONE agentic product with spiky traffic — the book's
# canonical "you do not need K8s yet" case.
verdict, score, reasons = recommend(
    services=1, teams=1, needs_gpu_scheduling=False, needs_multicloud=False,
    has_platform_engineer=False, traffic_is_spiky_but_simple=True,
)
print("Scenario: 2-person team, 1 agent service, spiky-but-simple traffic.")
for r in reasons:
    print("  -", r)
print(f"\nK8s score: {score}  ->  Recommend: {verdict}")
assert "Fargate" in verdict, "small single-service team should NOT adopt K8s here"

**What you just saw.** Capability isn't the question — *operational cost relative to your team* is. The same tool flips to "Kubernetes" once you genuinely have many services, multiple teams, GPU scheduling, or multi-cloud needs **and** someone who can operate it. Run it against your real numbers in the exercises.

## 🎯 Senior lens (§35.4)

The through-line of this whole part is **operational leverage versus operational cost**, and Kubernetes is its sharpest example: extraordinarily powerful, *and* a permanent investment in expertise and toil. The senior move is to be honest about your team's size and needs. A two-person team shipping an agentic product almost certainly wants **managed containers (Fargate)** and to spend its scarce attention on the *product*; a platform team serving many internal teams may genuinely need Kubernetes' flexibility.

Choose the level of infrastructure your team can **operate well**, not the most powerful one available — power you can't operate is a liability, not an asset. And notice: the unit of deploy is the *same image either way* (35-01). Getting the container right buys you the option to defer — and even reverse — the K8s decision without re-engineering the app.

## Recap

- Kubernetes is one idea — **declarative desired state** — and a reconcile loop that drives `running` toward `desired` forever. Self-healing and scaling are the same loop.
- The core objects each have one job: **Deployment** (keep N replicas), **Service** (stable address + LB), **Ingress** (front door), **HPA** (autoscale to a metric), **ConfigMap/Secret** (config & secrets injected at runtime).
- The **HPA** scales by ratio (`current × metric / target`), clamped to `[min, max]`; watch *where* you hit `maxReplicas` to size it and decide when to shed load.
- **readiness gates traffic; liveness triggers restart** — two different jobs; swapping them causes outages.
- **Helm** templates manifests across environments; **operators** automate day-two ops. Both are leverage *and* more to operate.
- The real skill is the **Fargate-vs-K8s** call: score operational *cost* against capability, and pick infra your team can operate well.

## Exercises

Each *changes something* and asks you to predict first.

1. **Scale-down debounce.** The reconciler above reacts instantly. Real HPAs *delay* scale-down to avoid thrashing on a brief dip. Add a `cooldown` so `replicas` only drops if the backlog has been low for N consecutive minutes. Predict how the worker count over the morning curve changes, then plot/print it.
2. **Your own Fargate-vs-K8s call.** Call `recommend(...)` with *your team's* real numbers (services, teams, GPU, multi-cloud, platform engineer). Predict the verdict before you run it. Which single factor would flip it the other way?
3. **Probe mix-up.** Change one timeline row so a *healthy-but-busy* Pod fails **liveness** instead of readiness. Predict what Kubernetes does, then run `pod_state` on it — and explain why this is a real outage pattern.
4. **Raise the cap.** Set `MAX_REPLICAS = 12` and re-run the HPA sim. Predict whether the peak `per-pod` backlog drops below 30, and what it costs you (Ch 31's idle-vs-latency trade-off).

In [ ]:
# Exercise 1 — add a scale-down cooldown to the reconcile/HPA loop.

In [ ]:
# Exercise 2 — recommend(...) with YOUR real numbers; predict the verdict first.

In [ ]:
# Exercise 3 — flip one probe row (busy Pod fails liveness); predict the action.

## Next

- **Previous notebook:** [`35-01-dockerize-a-service.ipynb`](35-01-dockerize-a-service.ipynb) — the image these manifests deploy (multi-stage, non-root, slim).
- **Template:** [`templates/dockerfile-and-compose/`](../../templates/dockerfile-and-compose/) — the container artifact from 35-01; the manifests here are the K8s-path reference for shipping it.
- **Book:** §35.2 (K8s core concepts + when you need it), §35.3 (deploying/scaling: HPA, probes), §35.4 (Helm, operators, the operational-cost lens).
- **Capstone:** these manifests are the K8s reference for the capstone's deploy story; they advance checkpoint `checkpoints/ch35-containerized`. **Ch 36 (Infrastructure as Code)** then provisions the cluster/managed-container target and deploys the 35-01 image.